# Day 2 - Lab 3: Feature engineering, framing and communicating

**Goal:** turn the cleaned table into a **model-ready** feature table for Day 3, then frame the business question and write the finding in plain English.

Feature engineering is where domain knowledge enters. A model is only as good as the columns you give it.

## 1. Load

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

def find_data(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('data/ folder not found')

DATA = find_data()

def _rebuild_clean():
    """Re-run the Day 1 cleaning from raw, so Day 2 is self-contained."""
    df = pd.read_csv(DATA / 'raw' / 'service_requests_raw.csv', dtype=str).drop_duplicates()
    df['priority'] = df['priority'].str.strip().str.title().replace({'2': 'Medium'})
    df['resolution_hours'] = pd.to_numeric(df['resolution_hours'], errors='coerce')
    iso   = pd.to_datetime(df['submitted_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    named = pd.to_datetime(df['submitted_at'], format='%d-%b-%Y %H:%M', errors='coerce')
    df['submitted_at'] = iso.fillna(named)
    df['resolved_at'] = pd.to_datetime(df['resolved_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df.loc[(df['resolution_hours'] < 0) | (df['resolution_hours'] > 8760), 'resolution_hours'] = np.nan
    df.loc[(df['resolved_at'] < df['submitted_at']).fillna(False), 'resolved_at'] = pd.NaT
    df['citizen_age_band'] = df['citizen_age_band'].replace({'': pd.NA, 'Unknown': pd.NA}).fillna('Unknown')
    df['service_id'] = pd.to_numeric(df['service_id'], errors='coerce')
    svc = pd.read_csv(DATA / 'seeds' / 'services.csv')
    df = df.merge(svc[['service_id', 'target_resolution_hours']], on='service_id', how='left')
    resolved = df['status'].isin(['Resolved', 'Reopened']) & df['resolution_hours'].notna()
    df['sla_met'] = pd.NA
    df.loc[resolved, 'sla_met'] = (df.loc[resolved, 'resolution_hours'] <= df.loc[resolved, 'target_resolution_hours']).astype('int')
    return df

def load_clean():
    p = DATA / 'processed' / 'service_requests_clean.csv'
    if p.exists():
        print('Loaded cleaned dataset from Day 1:', p.name)
        return pd.read_csv(p, parse_dates=['submitted_at', 'resolved_at'])
    print('Cleaned file not found, rebuilding from raw (Day 1 cleaning)...')
    return _rebuild_clean()

df = load_clean()
channels = pd.read_csv(DATA / 'seeds' / 'channels.csv')
districts = pd.read_csv(DATA / 'seeds' / 'districts.csv')
df = df.merge(channels, on='channel_id', how='left')
print('shape:', df.shape)
df.head(3)

## 2. Temporal features
A single timestamp hides several useful signals. Pull them out: the hour of day, the day of week, the month, and whether it was a weekend.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

## 3. Flags and ratios
`is_digital` is already a flag. A more expressive feature is the **SLA ratio**: resolution time divided by the target. Above 1 means the target was missed, and by how much.

In [ ]:
# TODO: your code here

## 4. Join for context: district population
The district a request came from carries context. Join the population so the model can learn whether busier districts behave differently.

In [ ]:
# TODO: your code here

## 5. Encode categoricals
Models need numbers. `priority` is **ordinal** (Low < Medium < High), so map it to a rank. Unordered categories like status would use one-hot encoding (`pd.get_dummies`) instead.

In [ ]:
# TODO: your code here

## 6. Assemble the model-ready table
Select the target and the features Day 3 will use, and save it. Keep the rows where the target is known for supervised learning.

In [ ]:
# TODO: your code here

## 7. Frame the business problem
Before Day 3 touches a model, state the problem in one sentence a manager would accept:

> *Can we predict, at the moment a request is logged, whether it will miss its SLA, so supervisors can intervene early?*

- **Target:** `sla_met` (did it meet the target).
- **Available at logging time:** channel, priority, service target, district, time of day. (Note `resolution_hours` is **not** known at logging time; it is the outcome. Day 3 will treat it carefully to avoid leakage.)
- **Value:** a supervisor who knows a request is at risk can reprioritise it before it slips.

## 8. Communicate the finding
One structured paragraph beats a wall of charts. Use: **what we found, how confident, what to do, what to check next.**

> *Digital channels resolve requests about 15 hours faster at the median than the call centre and walk-in centre, and meet their SLA far more often (around 60% versus 35%). The gap is large and statistically robust (p < 0.001). Low-priority requests miss their SLA most often (only 38% met), suggesting they are deprioritised until they slip. **Recommendation:** review how low-priority and non-digital requests are queued. **To check next:** whether non-digital requests are simply harder cases, rather than worse-handled ones.*

## Your turn
1. Add a feature `hours_since_midnight` or bucket `submitted_hour` into morning / afternoon / evening / night. Which is more useful, and why?
2. One-hot encode `channel_name` with `pd.get_dummies`. How many columns does it add?
3. Rewrite the finding paragraph for a **technical** Day 3 audience instead of a manager. What changes?

In [ ]:
# your turn